### create individual contributor summary statistics

In [ ]:
import os, sys
import pandas as pd
import sqlite3
import csv
import numpy as np
from scipy import stats
from google.cloud import bigquery
from google.cloud import storage

In [ ]:
# Set path
ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "data"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

In [ ]:
# Credentials 
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not credentials_path:
    raise EnvironmentError(
        "GOOGLE_APPLICATION_CREDENTIALS is not set. "
        "Please set it to the path of your Google Cloud credentials JSON file."
    )

credentials_path = Path(credentials_path).expanduser().resolve()

if not credentials_path.exists():
    raise FileNotFoundError(
        f"Google Cloud credentials file not found: {credentials_path}"
    )

In [ ]:
# Construct a BigQuery client object.
client = bigquery.Client()
storage_client = storage.Client()

In [ ]:
def check_costs(query):
    # settings
    dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
    run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

    # check processing costs
    job = client.query(query, job_config=dry)
    print("Total GB that will be processed: ", job.total_bytes_processed/1000000000)
    print("Bytes billed: ", job.total_bytes_billed)

In [ ]:
# settings for bigquery 
dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

In [ ]:
# count total number of users/contributors in ua, ru, control in 2011 - 2018

query = """
SELECT 
  COUNT(DISTINCT CASE WHEN country_author = 'ua' THEN author_id END) AS ua_users_18,
  COUNT(DISTINCT CASE WHEN country_author = 'ru' THEN author_id END) AS ru_users_18,
  COUNT(DISTINCT CASE 
                   WHEN country_author NOT IN ('ua','ru') AND
                        country_author IN  ('us', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'pl', 'jp', 'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee')
                   THEN author_id 
                 END) AS control_users_18
FROM `friends-foes.commits.puc` 

LEFT JOIN
  (SELECT id, EXTRACT(YEAR FROM created_at) AS year_signup
  FROM `ghtorrentmysql1906.MySQL1906.users`)
ON author_id = id

WHERE country_author IN ('ua', 'ru', 'us', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'pl', 'jp', 'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee')
  AND author_id <> owner_id 
  AND year_signup < 2019;

"""

check_costs(query)
df_users_18 = client.query(query, job_config=run).to_dataframe()

In [ ]:
# count total number of users/contributors in ua, ru, control pre 2014

query = """
SELECT 
  COUNT(DISTINCT CASE WHEN country_author = 'ua' THEN author_id END) AS ua_users_14,
  COUNT(DISTINCT CASE WHEN country_author = 'ru' THEN author_id END) AS ru_users_14,
  COUNT(DISTINCT CASE 
                   WHEN country_author NOT IN ('ua','ru') AND
                        country_author IN  ('us', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'pl', 'jp', 'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee')
                   THEN author_id 
                 END) AS control_users_14
FROM `friends-foes.commits.puc` 

LEFT JOIN
  (SELECT id, EXTRACT(YEAR FROM created_at) AS year_signup
  FROM `ghtorrentmysql1906.MySQL1906.users`)
ON author_id = id

WHERE country_author IN ('ua', 'ru', 'us', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'pl', 'jp', 'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee')
  AND author_id <> owner_id 
  AND year_signup < 2014;

"""

check_costs(query)
df_users_14 = client.query(query, job_config=run).to_dataframe()

In [ ]:
# preview summary statistics
print(df_users_14.ua_users_14)
print(df_users_14.ru_users_14)
print(df_users_14.control_users_14)
total_users = df_users_14.ua_users_14 + df_users_14.ru_users_14 + df_users_14.control_users_14
print(total_users)

In [ ]:
# contributor-level summary statistics - commits

query = """
CREATE OR REPLACE TABLE  `friends-foes.commits.contributors_summary_statistics` AS

SELECT 
  author_id,
  country_author,
  SUM(commits) AS commits_pre2014,
  SUM(CASE WHEN country_owner = "ua" THEN commits ELSE 0 END) AS commits_ua_pre2014,
  SUM(CASE WHEN country_owner = "ru" THEN commits ELSE 0 END) AS commits_ru_pre2014,
  COUNT(DISTINCT owner_id) AS owners_pre2014,
  COUNT(DISTINCT country_owner) AS countries_pre2014
  
FROM `friends-foes.commits.puc` 

WHERE country_author IN ('ua', 'ru', 'us', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'pl', 'jp', 'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee')
  AND author_id <> owner_id 
  AND year < 2014
  
GROUP BY author_id, country_author
ORDER BY country_author, author_id
"""
check_costs(query)
client.query(query, job_config=run)

In [ ]:
# add number of projects owned before 2014

query = """
CREATE OR REPLACE TABLE `friends-foes.commits.contributors_summary_extended` AS
WITH base AS (
  SELECT *
  FROM `friends-foes.commits.contributors_summary`
),

-- Pre-aggregate projects owned before 2014
projects AS (
  SELECT 
    owner_id,
    COUNT(DISTINCT id) AS projects_pre2014
  FROM `ghtorrentmysql1906.MySQL1906.projects`
  WHERE EXTRACT(YEAR FROM created_at) < 2014
  GROUP BY owner_id
),

-- Pre-aggregate followers before 2014
followers AS (
  SELECT 
    user_id,
    COUNT(DISTINCT follower_id) AS followers_pre2014
  FROM `ghtorrentmysql1906.MySQL1906.followers`
  WHERE EXTRACT(YEAR FROM created_at) < 2014
  GROUP BY user_id
),

-- Get GitHub signup year
users AS (
  SELECT 
    id AS user_id,
    EXTRACT(YEAR FROM created_at) AS signup_gh
  FROM `ghtorrentmysql1906.MySQL1906.users`
)

SELECT
  b.author_id,
  b.country_author,
  b.commits_pre2014,
  b.commits_ua_pre2014,
  b.commits_ru_pre2014,
  b.owners_pre2014,
  b.countries_pre2014,
  COALESCE(p.projects_pre2014, 0) AS projects_pre2014,
  COALESCE(f.followers_pre2014, 0) AS followers_pre2014,
  u.signup_gh
FROM base b
LEFT JOIN projects p
  ON b.author_id = p.owner_id
LEFT JOIN followers f
  ON b.author_id = f.user_id
LEFT JOIN users u
  ON b.author_id = u.user_id

ORDER BY author_id, country_author
"""

check_costs(query)
client.query(query, job_config=run)

In [ ]:
query = '''
SELECT * FROM `friends-foes.commits.contributors_summary_extended`
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(DATA / 'users_summary_statistics.csv', encoding = 'utf-8', index=False)


In [ ]:
# create summary statistics table

# --- Define groups ---
df["group"] = np.where(df["country_author"]=="ua", "UA",
                np.where(df["country_author"]=="ru", "RU", "Control"))

# --- Variables and labels ---
vars_of_interest = [
    "commits_pre2014",
    "commits_ua_pre2014",
    "commits_ru_pre2014",
    "owners_pre2014",
    "countries_pre2014",
    "projects_pre2014",
    "followers_pre2014",
    "signup_gh"
]

labels = {
    "commits_pre2014": "Commits pre 2014",
    "commits_ua_pre2014": "Commits to UA projects pre 2014",
    "commits_ru_pre2014": "Commits to RU projects pre 2014",
    "owners_pre2014": "Project owners collaborated with pre 2014",
    "countries_pre2014": "Countries collaborated with pre 2014",
    "projects_pre2014": "Projects owned pre 2014",
    "followers_pre2014": "Followers pre 2014",
    "signup_gh": "Year of GitHub registration pre 2014"
}

# --- Container for results ---
rows = []

for v in vars_of_interest:
    # Means
    mean_ua = df.loc[df["group"]=="UA", v].mean()
    mean_ru = df.loc[df["group"]=="RU", v].mean()
    mean_ctrl = df.loc[df["group"]=="Control", v].mean()
    
    # UA vs RU Welch t-test
    ua_vals = df.loc[df["group"]=="UA", v].dropna()
    ru_vals = df.loc[df["group"]=="RU", v].dropna()
    tstat, pval = stats.ttest_ind(ua_vals, ru_vals, equal_var=False)
    diff = mean_ua - mean_ru
    
    # Significance stars
    if pval < 0.01:
        stars = "***"
    elif pval < 0.05:
        stars = "**"
    elif pval < 0.10:
        stars = "*"
    else:
        stars = ""
    
    # Append row
    rows.append([labels[v], mean_ua, mean_ru, mean_ctrl, diff, pval, stars])

# --- Results DataFrame ---
res = pd.DataFrame(rows, columns=["Variable", "UA", "RU", "Control", "UA-RU", "p-value", "Significance"])

# --- Round means and differences to 1 decimal, p-values to 3 decimals ---
res[["UA", "RU", "Control", "UA-RU"]] = res[["UA", "RU", "Control", "UA-RU"]].round(1)
res["p-value"] = res["p-value"].round(2)

# --- Add sample sizes as last row (no decimals) ---
n_ua = df.loc[df["group"]=="UA"].shape[0]
n_ru = df.loc[df["group"]=="RU"].shape[0]
n_ctrl = df.loc[df["group"]=="Control"].shape[0]

res.loc[len(res)] = ["Users pre 2019", int(df_users_18.ua_users_18), int(df_users_18.ru_users_18), int(df_users_18.control_users_18), "", "", ""]
res.loc[len(res)] = ["Users pre 2014", int(df_users_14.ua_users_14), int(df_users_14.ru_users_14), int(df_users_14.control_users_14), "", "", ""]
#res.loc[len(res)] = ["Users pre 2014", int(n_ua), int(n_ru), int(n_ctrl), "", "", ""]

# Export to LaTeX
with open(TABLES / "01_user_summary.tex", "w") as f:
    f.write(res.to_latex(index=False, 
                         escape=False,    # allow stars
                         column_format="lcccccc", 
                         caption="User Summary Statistics",
                         label="tab:user_summary_diffs",
                         longtable=False))
